In [4]:
import duckdb

con = duckdb.connect('f1.duckdb')

con.execute("SELECT * FROM driver_season").df()

,driverId,year,total_points,races_entered,dnfs,avg_grid,avg_finish,completed_races,ref,constructor
0,3,2014,317.0,NaN,2,1.684211,2.529412,16,rosberg,131
1,825,2014,55.0,NaN,1,8.789474,9.388889,12,kevin_magnussen,1
2,18,2014,126.0,NaN,1,8.473684,7.500000,13,button,1
3,4,2014,161.0,NaN,2,6.526316,5.411765,17,alonso,6
4,822,2014,186.0,NaN,1,6.210526,5.555556,17,bottas,3
...,...,...,...,...,...,...,...,...,...,...
239,154,2020,2.0,NaN,3,14.133333,14.666667,6,grosjean,210
240,20,2020,33.0,NaN,2,12.058824,10.400000,9,vettel,6
241,857,2023,82.0,NaN,3,9.636364,9.631579,14,piastri,1
242,840,2024,24.0,NaN,1,11.615385,11.166667,8,stroll,117


In [5]:
con.execute("SHOW TABLES").df()

,name
0,circuits
1,constructor_results
2,constructor_standings
3,constructors
4,driver_season
5,driver_standings
6,drivers
7,lap_times
8,pit_stops
9,qualifying


I forgot to populate the races entered column

In [7]:
con = duckdb.connect("f1.duckdb")

con.execute(
"UPDATE driver_season ds "
"SET races_entered = agg.races_entered "
"FROM ( "
"    SELECT driverId, year, "
"           COUNT(DISTINCT raceId) AS races_entered "
"    FROM results_drivers_races_circuits "
"    WHERE year >= 2014 "
"    GROUP BY driverId, year "
") agg "
"WHERE ds.driverId = agg.driverId "
"AND ds.year = agg.year "
)

con.execute("SELECT * FROM driver_season").df()

,driverId,year,total_points,races_entered,dnfs,avg_grid,avg_finish,completed_races,ref,constructor
0,3,2014,317.0,19,2,1.684211,2.529412,16,rosberg,131
1,825,2014,55.0,19,1,8.789474,9.388889,12,kevin_magnussen,1
2,18,2014,126.0,19,1,8.473684,7.500000,13,button,1
3,4,2014,161.0,19,2,6.526316,5.411765,17,alonso,6
4,822,2014,186.0,19,1,6.210526,5.555556,17,bottas,3
...,...,...,...,...,...,...,...,...,...,...
239,154,2020,2.0,15,3,14.133333,14.666667,6,grosjean,210
240,20,2020,33.0,17,2,12.058824,10.400000,9,vettel,6
241,857,2023,82.0,22,3,9.636364,9.631579,14,piastri,1
242,840,2024,24.0,13,1,11.615385,11.166667,8,stroll,117


## Driver reliability

These features main purpose is to represent the factors that come into play that are influeced by a driver, it is expected to have an overlap with non-driver factors.

Starting i will create a dnf rate feature, this indicates how many races in a season a driver does not finish. This is one of the most basic explanatory features

In [41]:
con = duckdb.connect("f1.duckdb")

con.execute("CREATE TABLE features AS "
"SELECT "
    "ds.driverId, "
    "ds.year, "
    "SUM(CASE WHEN r.milliseconds IS NULL THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS DNF_rate "
" FROM driver_season ds "
"JOIN results_drivers_races_circuits r "
    "ON ds.driverId = r.driverId AND ds.year = r.year "
"GROUP BY ds.driverId, ds.year;")


In [43]:
con.execute("DESCRIBE driver_season").df()

,column_name,column_type,null,key,default,extra
0,driverId,BIGINT,YES,NaN,NaN,NaN
1,year,BIGINT,YES,NaN,NaN,NaN
2,total_points,DOUBLE,YES,NaN,NaN,NaN
3,races_entered,BIGINT,YES,NaN,NaN,NaN
4,dnfs,BIGINT,YES,NaN,NaN,NaN
5,avg_grid,DOUBLE,YES,NaN,NaN,NaN
6,avg_finish,DOUBLE,YES,NaN,NaN,NaN
7,completed_races,BIGINT,YES,NaN,NaN,NaN
8,constructor,VARCHAR,YES,NaN,NaN,NaN
9,ref,VARCHAR,YES,NaN,NaN,NaN


In [46]:
con = duckdb.connect("f1.duckdb")
con.execute("ALTER TABLE features "
"ADD COLUMN ref VARCHAR;")

In [47]:
con.execute("ALTER TABLE features "
"ADD COLUMN constructor VARCHAR;")

In [55]:
con.execute("UPDATE features f "
"SET ref = ds.ref "
"FROM driver_season ds "
"WHERE f.driverId = ds.driverId "
    "AND f.year = ds.year;")

In [56]:
con.execute("UPDATE features f "
"SET constructor = ds.constructor "
"FROM driver_season ds "
"WHERE f.driverId = ds.driverId "
    "AND f.year = ds.year;")


In [59]:
con.execute("ALTER TABLE features "
"ADD COLUMN constructorRef VARCHAR;")

In [61]:
con.execute("UPDATE features f "
"SET constructorRef = c.constructorRef "
"FROM constructors c "
"WHERE f.constructor = c.constructorId;")
con.close()

In [63]:
con = duckdb.connect("f1.duckdb")

con.execute("SELECT * FROM features").df()

,driverId,year,DNF_rate,ref,constructor,constructorRef
0,3,2014,0.157895,rosberg,131,mercedes
1,825,2014,0.368421,kevin_magnussen,1,mclaren
2,18,2014,0.315789,button,1,mclaren
3,4,2014,0.105263,alonso,6,ferrari
4,822,2014,0.105263,bottas,3,williams
...,...,...,...,...,...,...
239,859,2023,0.400000,lawson,213,alphatauri
240,20,2020,0.470588,vettel,6,ferrari
241,825,2023,0.681818,kevin_magnussen,210,haas
242,840,2024,0.384615,stroll,117,aston_martin


DNF rate is a very powerful explanatory variable at least on paper, it is a direct indicator of points and we expect drivers with a high dnf rate to have a low point score. This feature captures both the drivers mistake, and the constructor caused DNFS. 

To capture driver performance beyond car pace, I intend to create a feature called position improvement, defined as the difference between average starting grid position and average finishing position at the season level. This metric acts as a representation of a driver's in-race performance.

Since average grid position was calculated for all races including those in which DNFS were the resulting position, computing this improvement with that feature would introduce bias, which is why I will recalculate so that the position improvement is calculated ONLY in races for which there are NO DNFS, as DNF will already by accounted for in prediction.

In [64]:
con.execute(
"UPDATE driver_season ds "
"SET avg_grid = agg.avg_grid "
"FROM ( "
"    SELECT driverId, year, "
"           AVG(grid) AS avg_grid "
"    FROM results_drivers_races_circuits "
"    WHERE position IS NOT NULL "
"      AND grid IS NOT NULL "
"    GROUP BY driverId, year "
") agg "
"WHERE ds.driverId = agg.driverId "
"AND ds.year = agg.year "
)
con.close()

In [71]:
con = duckdb.connect("f1.duckdb")

con.execute("SELECT (avg_grid) FROM driver_season ").df()

,avg_grid
0,1.705882
1,8.833333
2,8.333333
3,6.588235
4,5.833333
...,...
239,13.416667
240,11.866667
241,9.421053
242,11.750000


In [72]:
con.close()

position_delta = avg_grid - avg_finish. If a driver's position delta is positive then it means that on average, he improves a position_delta number of positions on a race, and if it is negative, it means that on average, he worsens a position_delta number of positions on a race.

In [76]:
con = duckdb.connect("f1.duckdb")
con.execute("ALTER TABLE features "
"ADD COLUMN position_delta double;")

In [78]:
con.execute(
"UPDATE features f "
"SET position_delta = ds.avg_grid - ds.avg_finish "
"FROM driver_season ds "
"WHERE f.driverId = ds.driverId "
"AND f.year = ds.year "
)

In [80]:
con.execute("SELECT (*) FROM features").df()

,driverId,year,DNF_rate,ref,constructor,constructorRef,position_delta
0,3,2014,0.157895,rosberg,131,mercedes,-0.823529
1,825,2014,0.368421,kevin_magnussen,1,mclaren,-0.555556
2,18,2014,0.315789,button,1,mclaren,0.833333
3,4,2014,0.105263,alonso,6,ferrari,1.176471
4,822,2014,0.105263,bottas,3,williams,0.277778
...,...,...,...,...,...,...,...
239,859,2023,0.400000,lawson,213,alphatauri,1.600000
240,20,2020,0.470588,vettel,6,ferrari,1.466667
241,825,2023,0.681818,kevin_magnussen,210,haas,-3.111111
242,840,2024,0.384615,stroll,117,aston_martin,0.583333


Driver's with a high DNF rate may have a strong position delta since their results are gonna be based on a few races.

In [88]:
con = duckdb.connect("f1.duckdb")

high_dnf = con.execute(
    "SELECT * FROM features WHERE DNF_rate > 0.30"
).df()

high_dnf

,driverId,year,DNF_rate,ref,constructor,constructorRef,position_delta
0,825,2014,0.368421,kevin_magnussen,1,mclaren,-0.555556
1,18,2014,0.315789,button,1,mclaren,0.833333
2,8,2014,0.368421,raikkonen,6,ferrari,-0.277778
3,818,2014,0.473684,vergne,5,toro_rosso,0.785714
4,826,2014,0.684211,kvyat,5,toro_rosso,0.428571
...,...,...,...,...,...,...,...
165,859,2023,0.400000,lawson,213,alphatauri,1.600000
166,20,2020,0.470588,vettel,6,ferrari,1.466667
167,825,2023,0.681818,kevin_magnussen,210,haas,-3.111111
168,840,2024,0.384615,stroll,117,aston_martin,0.583333


In [87]:
con.close()

This feature on itself is going to be misleading, but paired with DNF rate will give the model a broader understanding of driver performance, so it's essential to keep this in mind moving forward.

It is very important to have in mind that the position delta for very dominant drivers (such as Hamilton 2014-2020) is expected to be near 0 since they usually start in very high positions and end in the same position, and if there are any changes in their finish position in respect to their grid position it is going to be low, this makes it crucial for this feature to be used alongside other explanatory variables which will give more context to the model. I considered normalizing this feature, but for a xgboost model, or even a linear model with other explanatory variables, this ceiling will be understood by the model.

High altitude is an important factor from a mechanical aspect, so I intend to create a feature that captures how good a driver perform in these high altitude settings which usually represent a more difficult scenario for a driver in a race.

Moving forward I will define high altitude as any altitude that is of 500m AND above.

In [92]:
con = duckdb.connect("f1.duckdb")
high_alt_circuits = con.execute("SELECT * FROM circuits WHERE alt >= 500; ").df()
high_alt_circuits

,circuitId,circuitRef,name,location,country,lat,lng,alt,url
0,16,fuji,Fuji Speedway,Oyama,Japan,35.3717,138.92700,583,http://en.wikipedia.org/wiki/Fuji_Speedway
1,18,interlagos,Autódromo José Carlos Pace,São Paulo,Brazil,-23.7036,-46.69970,785,http://en.wikipedia.org/wiki/Aut%C3%B3dromo_Jo...
2,20,nurburgring,Nürburgring,Nürburg,Germany,50.3356,6.94750,578,http://en.wikipedia.org/wiki/N%C3%BCrburgring
3,80,vegas,Las Vegas Strip Street Circuit,Las Vegas,United States,36.1147,-115.17300,642,https://en.wikipedia.org/wiki/Las_Vegas_Grand_...
4,30,kyalami,Kyalami,Midrand,South Africa,-25.9894,28.07670,1460,http://en.wikipedia.org/wiki/Kyalami
5,32,rodriguez,Autódromo Hermanos Rodríguez,Mexico City,Mexico,19.4042,-99.09070,2227,http://en.wikipedia.org/wiki/Aut%C3%B3dromo_He...
6,36,jacarepagua,Autódromo Internacional Nelson Piquet,Rio de Janeiro,Brazil,-22.9756,-43.39500,1126,http://en.wikipedia.org/wiki/Aut%C3%B3dromo_In...
7,44,las_vegas,Las Vegas Street Circuit,Nevada,USA,36.1162,-115.17400,639,http://en.wikipedia.org/wiki/Las_Vegas_Street_...
8,45,jarama,Jarama,Madrid,Spain,40.6171,-3.58558,609,http://en.wikipedia.org/wiki/Circuito_Permanen...
9,51,charade,Charade Circuit,Clermont-Ferrand,France,45.7472,3.03889,790,http://en.wikipedia.org/wiki/Charade_Circuit


In [96]:
#High altitude race results per driver
con.execute(
"SELECT r.driverId, r.year, r.circuitId, r.position, r.grid "
"FROM results_drivers_races_circuits r "
"JOIN circuits c "
  "ON r.circuitId = c.circuitId "
"WHERE c.alt >= 500 AND r.year >= 2014;").df()

,driverId,year,circuitId,position,grid
0,3,2014,18,1.0,1
1,1,2014,18,2.0,2
2,13,2014,18,3.0,3
3,18,2014,18,4.0,5
4,20,2014,18,5.0,6
...,...,...,...,...,...
641,840,2021,18,NaN,14
642,842,2021,18,7.0,7
643,839,2021,18,8.0,8
644,852,2021,18,15.0,15


In [97]:
con.execute("ALTER TABLE features "
"ADD COLUMN high_alt_sucess_rate double;")

In [99]:
# High-altitude success rate per driver-season
high_alt_success_rate = con.execute(
"SELECT "
"    r.driverId, "
"    r.year, "
"    CAST(SUM(CASE WHEN r.position IS NOT NULL AND r.position <= ds.avg_finish THEN 1 ELSE 0 END) AS FLOAT) / COUNT(r.position) AS high_alt_success_rate "
"FROM results_drivers_races_circuits r "
"JOIN circuits c "
"    ON r.circuitId = c.circuitId "
"JOIN driver_season ds "
"    ON r.driverId = ds.driverId AND r.year = ds.year "
"WHERE r.year >= 2014 AND c.alt >= 500 "
"GROUP BY r.driverId, r.year;"
).df()
high_alt_success_rate

,driverId,year,high_alt_success_rate
0,8,2016,0.500000
1,3,2016,0.666667
2,817,2016,0.333333
3,18,2016,0.333333
4,832,2016,0.666667
...,...,...,...
225,829,2015,0.500000
226,20,2020,0.500000
227,857,2023,0.250000
228,154,2014,0.000000


In [102]:
con.execute(
"UPDATE features f "
"SET high_alt_sucess_rate = sub.high_alt_sucess_rate "
"FROM ( "
"    SELECT r.driverId, r.year, "
"           AVG(CASE "
"               WHEN r.position IS NOT NULL AND r.position <= ds.avg_finish THEN 1.0 "
"               ELSE 0.0 "
"           END) AS high_alt_sucess_rate "
"    FROM results_drivers_races_circuits r "
"    JOIN circuits c "
"      ON r.circuitId = c.circuitId "
"    JOIN driver_season ds "
"      ON r.driverId = ds.driverId AND r.year = ds.year "
"    WHERE r.year >= 2014 AND c.alt >= 500 "
"    GROUP BY r.driverId, r.year "
") sub "
"WHERE f.driverId = sub.driverId AND f.year = sub.year;"
)


In [ ]:
con.execute("SELECT (*) FROM features").df()

In [104]:
con.close()

Now we have a couple strong driver-oriented performance metrics engineered, due to the limitation of the data at hand, I will now move on to team-oriented performance metrics, although I may come back in the future to work on this.